In [2]:
%%writefile D:\iip_asset_pricing\setup.py
from setuptools import setup, find_packages

setup(
    name="esi_asset_pricing",
    version="0.1.0",
    description="Empirical Asset Pricing with Effective Soothing Index (ESI)",
    author="Your Name",
    packages=find_packages(),
)

Overwriting D:\iip_asset_pricing\setup.py


In [3]:
%cd D:\iip_asset_pricing
!pip install -e .

D:\iip_asset_pricing
Obtaining file:///D:/iip_asset_pricing
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for esi_asset_pricing (pyproject.toml): started
  Building editable for esi_asset_pricing (pyproject.toml): finished with status 'done'
  Created wheel for esi_asset_pricing: filename=esi_asset_pricing-0.1.0-0.editable-py3-none-any.whl size=2814 sha256=94d358b12f9c02a77f3d8355808e84929be5b6f9b35612ac4e1bbfdfabcaf811
  Stored in directory: C:\Users\Admin\AppData\Local\Temp\pip-ephem-wheel-cache-5isrx1tu\wheels\90\0f\b1\3f

In [5]:
# 1. 验证是否能成功导入我们刚刚写的模块
import pandas as pd
import numpy as np

try:
    from src.features.esi_builder import construct_esi_factor
    from src.evaluation.fast_ols import calculate_residuals_fast
    from src.models.sorters import PortfolioSorter # 如果你加了这个的话
    print("✅ 恭喜！src 模块全部导入成功，工程化架构生效！")
except ModuleNotFoundError as e:
    print(f"❌ 导入失败，请检查 pip install -e . 是否成功执行，或者文件夹有没有 __init__.py 文件。报错: {e}")

✅ Conditional Double Sorter Engine Loaded.
✅ 恭喜！src 模块全部导入成功，工程化架构生效！


In [6]:
# 2. 读取一小块真实数据进行测试（限制 20000 行，极速测试）
from pathlib import Path

# 指向你的终极面板路径
DATA_PATH = Path(r"D:\iip_asset_pricing\data\processed\01_Base_Daily_Panel_Advanced.parquet")

print("-> 正在读取极小部分数据用于测试...")
# 仅读取前 20000 行
df_mini = pd.read_parquet(DATA_PATH).head(20000) 

# 确保有下一期的收益率用于回归
df_mini['Next_ExRet_O2C'] = df_mini.groupby('Stkcd')['Dretwd'].shift(-1).fillna(0)

print(f"✅ 测试数据就绪，维度: {df_mini.shape}")

-> 正在读取极小部分数据用于测试...
✅ 测试数据就绪，维度: (20000, 63)


In [7]:
# 3. 测试 ESI 构建器
print("-> 测试 construct_esi_factor 函数...")
# 备份一下原始列数
cols_before = df_mini.shape[1]

# 运行函数
df_mini = construct_esi_factor(df_mini, gamma=5.0)

cols_after = df_mini.shape[1]
if cols_after > cols_before and 'ESI_Gamma_5.0' in df_mini.columns:
    print("✅ ESI 构建器测试通过！成功生成了 ESI_Gamma_5.0 列。")
else:
    print("⚠️ 运行完成，但没有生成新列，请检查数据中是否有缺失的文本信号。")

-> 测试 construct_esi_factor 函数...
✅ ESI 构建器测试通过！成功生成了 ESI_Gamma_5.0 列。


In [8]:
# 4. 测试极速 OLS 残差计算器
print("-> 测试 calculate_residuals_fast 函数 (Numpy 极限加速)...")

# 随便挑几个因子作为 X (确保你的数据里有这几列)
test_x_cols = ['q5_MKT', 'q5_ME', 'q5_IA']
y_col = 'Next_ExRet_O2C'

# 填充一下因子里的 NA 防止报错
df_mini[test_x_cols] = df_mini[test_x_cols].fillna(0)

# 运行 OLS
df_mini = calculate_residuals_fast(df_mini, y_col=y_col, x_cols=test_x_cols, res_col_name='Test_ARet')

if 'Test_ARet' in df_mini.columns:
    print("✅ 极速 OLS 测试通过！成功计算出 Test_ARet 残差。")
    display(df_mini[['Stkcd', 'Date', y_col, 'Test_ARet']].dropna().head())
else:
    print("❌ OLS 运行失败。")

-> 测试 calculate_residuals_fast 函数 (Numpy 极限加速)...


Calculating Test_ARet:   0%|          | 0/6 [00:00<?, ?it/s]

✅ 极速 OLS 测试通过！成功计算出 Test_ARet 残差。


,Stkcd,Date,Next_ExRet_O2C,Test_ARet
0,000001,2010-01-04,-0.017292,-0.017586
1,000001,2010-01-05,-0.017167,-0.017461
2,000001,2010-01-06,-0.010917,-0.011257
3,000001,2010-01-07,-0.002208,-0.003703
4,000001,2010-01-08,0.000000,-0.001041
